# 📊 Exploratory Data Analysis (EDA): Academic Profiles & Student Performance
## Uncovering Latent Structures Prior to Dimensionality Reduction (PCA & t-SNE)

---

### Executive Summary & Objective
Raw academic grades often obscure inter-disciplinary strengths and vulnerabilities. A simple cumulative Grade Point Average (GPA) treats all subjects equally, frequently masking critical trade-offs—such as a student excelling in Humanities while falling critically behind in STEM, or inflated Elective marks masking core academic risk.

Before performing unsupervised dimensionality reduction (**Principal Component Analysis** & **t-SNE / UMAP**), this notebook conducts a rigorous, end-to-end **Exploratory Data Analysis (EDA)** on the `student_grades.csv` dataset across 8 subject areas:
* **STEM:** `math`, `science`, `cs`
* **Humanities:** `english`, `history`, `spanish`
* **Electives:** `band`, `physed`

### Goals of this EDA:
1. **Data Hygiene & Integrity:** Verify zero data corruption, missing entries, or scale anomalies.
2. **Univariate Analysis:** Understand the central tendency, spread, skewness, and variance disparities across subjects.
3. **Bivariate & Block Correlations:** Mathematically prove the presence of collinear subject clusters (STEM vs. Humanities vs. Electives).
4. **Inter-Disciplinary Trade-off Discovery:** Profile academic trade-offs using composite domain scores.
5. **Outlier & Cohort Detection:** Identify distinctive student cohorts (at-risk, specialized, all-rounders).
6. **Feature Scaling Feasibility:** Assess why Z-score standardization (`StandardScaler`) is necessary before PCA / t-SNE.


---
## 1. Environment Setup & Library Imports
We import standard scientific computing, data manipulation, and visualization libraries (`pandas`, `numpy`, `matplotlib`, `seaborn`, `scipy`). We also configure high-resolution visualization themes and robust path resolution.


In [ ]:
import os
import sys
import pathlib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set styling aesthetics
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.05)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42

print("Libraries imported successfully!")


---
## 2. Robust Data Loading & Data Quality Audit
To guarantee seamless execution across local VS Code kernels, standard Jupyter notebooks, and cloud environments (Colab), we implement dynamic path resolution.


In [ ]:
# Dynamically resolve dataset path
def find_dataset(filename="student_grades.csv"):
    candidates = [
        pathlib.Path(filename),
        pathlib.Path.cwd() / filename,
        pathlib.Path.cwd().parent / filename,
        pathlib.Path(__file__).resolve().parent / filename if '__file__' in globals() else None,
        pathlib.Path(globals().get('__vsc_ipynb_file__', '')).parent / filename if '__vsc_ipynb_file__' in globals() else None
    ]
    for p in candidates:
        if p and p.is_file():
            return str(p)
    return filename

csv_path = find_dataset("student_grades.csv")
df = pd.read_csv(csv_path)

# Define feature groups
STEM_COLS = ["math", "science", "cs"]
HUMANITIES_COLS = ["english", "history", "spanish"]
ELECTIVES_COLS = ["band", "physed"]
ALL_SUBJECTS = STEM_COLS + HUMANITIES_COLS + ELECTIVES_COLS

print(f"Loaded dataset from: {csv_path}")
print(f"Dataset Shape: {df.shape[0]} students × {df.shape[1]} columns\n")
df.head(10)


### 2.1 Missing Values, Data Types, and Duplicate Audit
We verify that each column matches expected numeric types and check for missing entries or duplicate student records.


In [ ]:
# Detailed Info & Missing Values
audit_df = pd.DataFrame({
    "Data Type": df.dtypes,
    "Null Count": df.isnull().sum(),
    "Null %": (df.isnull().sum() / len(df)) * 100,
    "Unique Values": df.nunique(),
    "Min": df.min(),
    "Max": df.max()
})

print("=== DATA QUALITY AUDIT ===")
display(audit_df)
print(f"\nDuplicate rows across all features: {df.duplicated().sum()}")
print(f"Duplicate student_ids: {df['student_id'].duplicated().sum()}")


**Key Observation:**
* The dataset contains exactly 100 students with zero missing values and zero duplicate records.
* Every subject grade is strictly bounded within a valid $[0, 100]$ percentage scale.


---
## 3. Univariate Statistical Profiling
Let's analyze the distribution metrics: mean, standard deviation, median, IQR, skewness, and kurtosis.


In [ ]:
stats_summary = df[ALL_SUBJECTS].describe().T
stats_summary["median"] = df[ALL_SUBJECTS].median()
stats_summary["IQR"] = stats_summary["75%"] - stats_summary["25%"]
stats_summary["skewness"] = df[ALL_SUBJECTS].skew()
stats_summary["kurtosis"] = df[ALL_SUBJECTS].kurtosis()

display(stats_summary[["mean", "std", "min", "25%", "median", "75%", "max", "IQR", "skewness", "kurtosis"]].round(2))


### 3.1 Grade Distribution Visualizations: Histograms & KDEs
We plot the distribution density for each subject grouped by academic domain.


In [ ]:
palette = {
    "STEM": "#1f77b4",
    "Humanities": "#e377c2",
    "Electives": "#2ca02c"
}

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

domain_map = {
    "math": "STEM", "science": "STEM", "cs": "STEM",
    "english": "Humanities", "history": "Humanities", "spanish": "Humanities",
    "band": "Electives", "physed": "Electives"
}

for i, col in enumerate(ALL_SUBJECTS):
    ax = axes[i]
    domain = domain_map[col]
    sns.histplot(df[col], kde=True, ax=ax, color=palette[domain], bins=15, edgecolor="black", alpha=0.6)
    ax.axvline(df[col].mean(), color="red", linestyle="--", linewidth=1.5, label=f"Mean: {df[col].mean():.1f}")
    ax.axvline(df[col].median(), color="black", linestyle=":", linewidth=1.5, label=f"Median: {df[col].median():.1f}")
    ax.set_title(f"{col.upper()} ({domain})", fontsize=12, fontweight="bold")
    ax.set_xlabel("Score (0 - 100)")
    ax.set_ylabel("Student Count")
    ax.legend(loc="upper left", fontsize=9)

# Hide 9th unused subplot
axes[8].set_visible(False)

plt.suptitle("Distribution of Student Grades Across 8 Subjects", fontsize=16, fontweight="bold", y=0.99)
plt.tight_layout()
plt.show()


### 3.2 Spread & Variance Disparity: Boxplots
Notice the notable variance differences between subject groups:
* **Humanities (`english`, `history`, `spanish`):** Large standard deviations ($\sigma pprox 17.3 - 18.4$) spanning from failing grades ($pprox 28$) to near-perfect ($pprox 99$).
* **STEM (`math`, `science`, `cs`):** Moderate standard deviations ($\sigma pprox 13.5 - 14.0$) with lower means ($pprox 53$).
* **Electives (`band`, `physed`):** Very tight distributions ($\sigma pprox 6.3 - 8.1$) with elevated means ($pprox 68 - 70$).

This fundamental variance disparity is the **primary mathematical justification for feature standardization prior to PCA**.


In [ ]:
plt.figure(figsize=(14, 6))
domain_colors = [palette[domain_map[col]] for col in ALL_SUBJECTS]

sns.boxplot(data=df[ALL_SUBJECTS], palette=domain_colors, width=0.5, fliersize=4)
sns.stripplot(data=df[ALL_SUBJECTS], color="black", alpha=0.3, jitter=0.2, size=4)

plt.title("Boxplot Comparison of Subject Distributions & Variance Disparities", fontsize=14, fontweight="bold")
plt.xlabel("Subject", fontsize=12)
plt.ylabel("Grade Score", fontsize=12)
plt.ylim(15, 105)

# Add custom legend for domains
import matplotlib.patches as mpatches
handles = [mpatches.Patch(color=color, label=dom) for dom, color in palette.items()]
plt.legend(handles=handles, title="Domain", loc="lower right")

plt.tight_layout()
plt.show()


---
## 4. Correlation & Collinearity Structure
Dimensionality reduction algorithms excel when high-dimensional data possesses strong underlying linear or non-linear correlations. Here, we calculate the Pearson correlation matrix to reveal latent modular blocks.


In [ ]:
# Compute Pearson correlation matrix
corr_matrix = df[ALL_SUBJECTS].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

sns.heatmap(
    corr_matrix, 
    annot=True, 
    fmt=".2f", 
    cmap="coolwarm", 
    vmin=-0.2, 
    vmax=1.0, 
    linewidths=1.2, 
    cbar_kws={"label": "Pearson Correlation (r)"}
)
plt.title("Pearson Correlation Heatmap Across All 8 Subjects", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


### 4.1 Block Correlation Matrix & Mathematical Proof of Clusters
The correlation matrix reveals three distinct blocks:
1. **STEM Block (`math`, `science`, `cs`):** Pairwise correlation $r \ge 0.97$. A student who excels in Math is almost guaranteed to excel in Science and CS.
2. **Humanities Block (`english`, `history`, `spanish`):** Pairwise correlation $r \ge 0.96$. Verbal/linguistic proficiency is tightly unified.
3. **Electives Block (`band`, `physed`):** Strong mutual correlation ($r = 0.80$).
4. **Cross-Domain Relationships:**
   - STEM vs. Humanities has moderate positive correlation ($r pprox 0.65 - 0.72$), reflecting general academic aptitude.
   - Electives show near-zero correlation with STEM and Humanities ($r pprox -0.04$ to $+0.13$). Elective ability is essentially orthogonal to core academic performance.


In [ ]:
# Display correlation table by block
stem_corr = df[STEM_COLS].corr().mean().mean()
hum_corr = df[HUMANITIES_COLS].corr().mean().mean()
elec_corr = df[ELECTIVES_COLS].corr().mean().mean()

core_elec_corr = df[STEM_COLS + HUMANITIES_COLS].corrwith(df["band"]).mean()

print(f"Average Intra-STEM Correlation:       {stem_corr:.3f}")
print(f"Average Intra-Humanities Correlation: {hum_corr:.3f}")
print(f"Average Intra-Electives Correlation:  {elec_corr:.3f}")
print(f"Average Core-to-Elective Correlation: {core_elec_corr:.3f} (Near Orthogonal!)")


---
## 5. Domain Composite Scores & Academic Trade-Off Analysis
We construct standardized composite scores for each domain:
* $\text{STEM Avg} = \frac{\text{Math} + \text{Science} + \text{CS}}{3}$
* $\text{Humanities Avg} = \frac{\text{English} + \text{History} + \text{Spanish}}{3}$
* $\text{Electives Avg} = \frac{\text{Band} + \text{PhysEd}}{2}$
* $\text{Overall Core GPA} = \frac{\text{STEM Avg} + \text{Humanities Avg}}{2}$


In [ ]:
df_domains = df.copy()
df_domains["STEM_Avg"] = df_domains[STEM_COLS].mean(axis=1)
df_domains["Humanities_Avg"] = df_domains[HUMANITIES_COLS].mean(axis=1)
df_domains["Electives_Avg"] = df_domains[ELECTIVES_COLS].mean(axis=1)
df_domains["Overall_Core_Avg"] = df_domains[STEM_COLS + HUMANITIES_COLS].mean(axis=1)
df_domains["STEM_minus_Hum"] = df_domains["STEM_Avg"] - df_domains["Humanities_Avg"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: STEM vs Humanities
sns.scatterplot(
    data=df_domains,
    x="Humanities_Avg",
    y="STEM_Avg",
    hue="Electives_Avg",
    palette="viridis",
    size="Overall_Core_Avg",
    sizes=(40, 200),
    ax=axes[0]
)
axes[0].plot([20, 100], [20, 100], "r--", alpha=0.7, label="Equal Aptitude Line (y = x)")
axes[0].set_title("STEM vs. Humanities: Identifying Domain Specialization", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Humanities Average Score", fontsize=11)
axes[0].set_ylabel("STEM Average Score", fontsize=11)
axes[0].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)

# Plot 2: Core Academic vs Electives
sns.scatterplot(
    data=df_domains,
    x="Overall_Core_Avg",
    y="Electives_Avg",
    hue="STEM_minus_Hum",
    palette="coolwarm",
    s=80,
    ax=axes[1]
)
axes[1].axhline(df_domains["Electives_Avg"].mean(), color="gray", linestyle=":", label="Electives Mean")
axes[1].axvline(df_domains["Overall_Core_Avg"].mean(), color="black", linestyle=":", label="Core Mean")
axes[1].set_title("Core Academic Average vs. Electives (Independence Proof)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Core Academic Average (Math, Sci, CS, Eng, Hist, Span)", fontsize=11)
axes[1].set_ylabel("Electives Average (Band, PhysEd)", fontsize=11)
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="STEM - Hum Diff", fontsize=9)

plt.tight_layout()
plt.show()


### 5.1 Pairwise Bivariate Relationships Across Core Clusters
We inspect the pairwise scatter distributions between representative features from each block (`math`, `english`, `band`).


In [ ]:
subset_features = ["math", "science", "english", "history", "band", "physed"]

g = sns.pairplot(
    df[subset_features],
    corner=True,
    diag_kind="kde",
    plot_kws={"alpha": 0.6, "edgecolor": "k", "s": 30}
)
g.fig.suptitle("Pairplot Across Cross-Domain Representative Subjects", y=1.02, fontsize=15, fontweight="bold")
plt.show()


---
## 6. Outlier Analysis & Cohort Identification
Using Z-Scores, Mahalanobis Distance, and IQR thresholds, we identify whether any extreme outliers exist, and profile natural academic archetypes.


In [ ]:
# Compute Z-Scores across subjects
z_scores = pd.DataFrame(stats.zscore(df[ALL_SUBJECTS]), columns=ALL_SUBJECTS)
outlier_mask = (np.abs(z_scores) > 2.5).any(axis=1)
outliers = df[outlier_mask]

print(f"Total students exceeding |Z| > 2.5 in any subject: {len(outliers)}")
print("Sample high-performing or extreme records:")
display(df.loc[outliers.index[:5], ["student_id"] + ALL_SUBJECTS])


### 6.1 Identifying the Four Student Archetypes
Based on our EDA findings, the student population naturally segregates into 4 distinct groups:
1. **High-Achieving All-Rounders:** Excel across all STEM ($>80$) and Humanities ($>85$).
2. **Humanities-Oriented Students:** Strong verbal/language marks ($>70$), but lower STEM performance ($<55$).
3. **Humanities & Electives Leaders:** High Humanities and exceptional participation in Band/PhysEd.
4. **At-Risk Cohort:** Struggling in core academics ($<45$), requiring immediate advising intervention.


In [ ]:
# Segment students into natural preliminary cohorts
conditions = [
    (df_domains["STEM_Avg"] >= 75) & (df_domains["Humanities_Avg"] >= 75),
    (df_domains["Humanities_Avg"] >= 65) & (df_domains["STEM_Avg"] < 65) & (df_domains["Electives_Avg"] >= 70),
    (df_domains["Humanities_Avg"] >= 65) & (df_domains["STEM_Avg"] < 65) & (df_domains["Electives_Avg"] < 70),
    (df_domains["Overall_Core_Avg"] < 50)
]
choices = [
    "High-Achieving All-Rounder",
    "Humanities & Electives Leader",
    "Humanities-Focused",
    "At-Risk / Foundational"
]

df_domains["Preliminary_Cohort"] = np.select(conditions, choices, default="Moderate All-Rounder")

cohort_counts = df_domains["Preliminary_Cohort"].value_counts()
print("=== Preliminary Student Cohorts (Pre-Clustering) ===")
display(cohort_counts)

plt.figure(figsize=(10, 5))
sns.barplot(x=cohort_counts.values, y=cohort_counts.index, palette="mako")
plt.title("Distribution of Student Academic Archetypes", fontsize=13, fontweight="bold")
plt.xlabel("Number of Students")
plt.ylabel("Archetype")
plt.tight_layout()
plt.show()


---
## 7. Scaling Feasibility: Why Standardization is Critical
Without standardization, PCA maximizes raw variance rather than correlation structure. Since `spanish` ($\sigma pprox 18.4$) has three times the variance of `band` ($\sigma pprox 6.3$), unscaled PCA would artificially over-weight Humanities.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_data = pd.DataFrame(scaler.fit_transform(df[ALL_SUBJECTS]), columns=ALL_SUBJECTS)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Raw Variance
df[ALL_SUBJECTS].var().plot(kind="bar", ax=axes[0], color="#d62728", edgecolor="black")
axes[0].set_title("Raw Feature Variance (Unscaled)", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Variance ($\sigma^2$)")
axes[0].set_xticklabels(ALL_SUBJECTS, rotation=45)

# Standardized Variance
scaled_data.var().plot(kind="bar", ax=axes[1], color="#2ca02c", edgecolor="black")
axes[1].set_title("Standardized Feature Variance (StandardScaler: $\\sigma^2 = 1.0$)", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Variance")
axes[1].set_xticklabels(ALL_SUBJECTS, rotation=45)

plt.suptitle("Variance Comparison: Demonstrating the Necessity of Preprocessing for PCA", fontsize=15, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()


---
## 8. EDA Synthesis & Bridge to Dimensionality Reduction

### Summary of Key Findings:
| Finding | Statistical Evidence | Impact on PCA / t-SNE |
| :--- | :--- | :--- |
| **Collinear Blocks** | $r_{\text{STEM}} \ge 0.97$, $r_{\text{Hum}} \ge 0.96$ | Data dimensionality can be compressed from 8D to 2D/3D with minimal information loss ($>95\%$ variance explained). |
| **Elective Orthogonality** | $r_{\text{Core, Electives}} \approx 0.05$ | Electives will form their own independent principal component (expected **PC2**). |
| **Variance Disparities** | Humanities $\sigma \approx 18$, Electives $\sigma \approx 6$ | Z-score scaling is non-negotiable to prevent Humanities from dominating the eigenvector calculation. |
| **Cohort Separation** | At-risk students ($n \approx 44$) cluster far from elite all-rounders ($n \approx 9$). | Non-linear manifold learning (t-SNE / UMAP) is expected to cleanly preserve these clusters in 2D. |

---
### Next Steps:
1. Proceed to **`01_PCA_Analysis.ipynb`** for Eigenvector decomposition, Scree plots, 2D/3D PCA visualization, and K-Means clustering.
2. Proceed to **`02_tSNE_Analysis.ipynb`** for hyperparameter tuning (perplexity, learning rate) and quantitative Silhouette score benchmarking against PCA.
